# Merging the Base Model with the DPO Adapter 

## Imports

In [1]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:    
    device = "cpu"

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODELS_DIR = Path("../models")
DPO_ADAPTER = MODELS_DIR / "dpo_model/dpo_final"
MERGER_PATH = MODELS_DIR / "final_model"

Using device: mps
PyTorch version: 2.11.0


In [2]:
tokenizer = AutoTokenizer.from_pretrained(
    str(DPO_ADAPTER),
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


## Loading the Base Model in float32
- Merging in bf16/fp16 can cause silent precision loss in the merged weights
- Load the Original Base Model and not the Adapter

In [3]:
print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype = torch.float32,
    trust_remote_code=True,
)

base_model.config.use_cache = False

print("Base model loaded successfully.")

Loading base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model loaded successfully.


## Loading the DPO Adapter
- load in `is_trainable=False` for merging
- also call .eval() before merging. This freezes the Adapter and lets PEFT cleanly fold the LoRA A/B Matrics into base models weights

In [7]:
print("Mounting DPO adapter...")

peft_model = PeftModel.from_pretrained(
    base_model,
    str(DPO_ADAPTER),
    is_trainable=False,
)

peft_model.eval()

print("DPO adapter mounted successfully.")
print(f"Adapter type: {peft_model.peft_type}")
print(f"Trainable parameters: {sum(p.numel() for p in peft_model.parameters() if p.requires_grad):,}")

Mounting DPO adapter...
DPO adapter mounted successfully.
Adapter type: PeftType.LORA
Trainable parameters: 0


## Merging LoRA Weights into base Model
- using `merge_and_upload()` 
- adds the LoRA Delta Weights into the base model 
- removes the LoRA adapter Structure entirely
- result is a HF causalLM - no PEFT

In [8]:
print("Merging adapter into base model weights...")

merged_model = peft_model.merge_and_unload()

print("Adapter merged successfully.")
print(f"Merged model type: {type(merged_model).__name__}")

# Verify that there are no remaining LoRA layers 
lora_layers = [
    n for n, _ in merged_model.named_modules()
    if "lora" in n.lower()
]

print(f"LoRA layers remaining after merge: {len(lora_layers)}")

Merging adapter into base model weights...
Adapter merged successfully.
Merged model type: Qwen2ForCausalLM
LoRA layers remaining after merge: 0


## Inference Call 

In [9]:
SYSTEM_PROMPT = (
    "You are a knowledgeable and empathetic customer support assistant. "
    "Your role is to help customers resolve their issues quickly and accurately. "
    "Always be polite, clear, and solution-focused."
)

def generate_response(model, tokenizer, user_query, max_ne_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    merged_model.to(device)
    merged_model.eval()
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=max_ne_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [11]:
test_queries = [
    "I want to cancel my subscription. How do I do that?",
    "I haven't received my order and it's been 10 days.",
    "How do I reset my account password?",
]

for q in test_queries:
    print(f"User query: {q}")
    response = generate_response(merged_model, tokenizer, q)
    print(f"Generated response: {response}\n")

User query: I want to cancel my subscription. How do I do that?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated response: Thank you for wanting to cancel your subscription! Cancellation of your subscription can be a bit like stepping out of a cozy room and finding yourself in an empty space. To cancel your subscription, we recommend following these steps:

1. **Access Your Account:** First, log into your account on our platform.
2. **Navigate to Subscription Settings:** Once logged in, look for the section labeled "Subscription" or similar.
3. **Select the Subscription:** Within the subscription settings, find the option related to cancellation.
4. **Make the Request:** Look for a button or link that says "Cancel Subscription" or something similar.
5. **Follow Instructions:** Follow the prompts provided by the system to complete the cancellation process.

If you encounter any difficulties or have specific questions during this process, please don't hesitate to let us know. We're here to assist you every step of the way in canceling your subscription. Enjoy the freedom of choosing when 

## Saving Merged Model

In [13]:
MERGER_PATH.mkdir(parents=True, exist_ok=True)

print(f"Saving merged model to {MERGER_PATH}...")

merged_model.save_pretrained(str(MERGER_PATH))
tokenizer.save_pretrained(str(MERGER_PATH))

print("Merged model and tokenizer saved successfully.")

print("\nSaved files:")
for f in sorted(MERGER_PATH.iterdir()):
    print(f"{f.name}")

Saving merged model to ../models/final_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model and tokenizer saved successfully.

Saved files:
chat_template.jinja
config.json
generation_config.json
model.safetensors
tokenizer.json
tokenizer_config.json


In [14]:
# Loading the merged model and tokenizer to verify they were saved correctly

verify_model = AutoModelForCausalLM.from_pretrained(
    str(MERGER_PATH),
    trust_remote_code=True,
    dtype=torch.float32,
)

total_params = sum(p.numel() for p in verify_model.parameters()) / 1e6

print(f"Verified merged model loaded successfully with {total_params:.2f}M parameters.")
print(f"Verified model type: {type(verify_model).__name__}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Verified merged model loaded successfully with 494.03M parameters.
Verified model type: Qwen2ForCausalLM


## Pushing Model to HUGGINGFACE MODELS

In [15]:
import os
from huggingface_hub import login

HF_USERNAME = "Ghostraptor"
HF_REPO_NAME = "qwen2.5-0.5b-customer-support-LoRA-dpo-merged"
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"Pushing merged model to: https://huggingface.co/{repo_id}...")

merged_model.push_to_hub(
    repo_id,
    commit_message = "Merged Qwen2.5-0.5B SFT+DPO customer support model"
)
tokenizer.push_to_hub(
    repo_id,
    commit_message = "Added tokenizer for merged Qwen2.5-0.5B SFT+DPO customer support model"
)

print(f"Model and tokenizer pushed to Hugging Face Hub successfully at https://huggingface.co/{repo_id}")

Pushing merged model to: https://huggingface.co/Ghostraptor/qwen2.5-0.5b-customer-support-LoRA-dpo-merged...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model and tokenizer pushed to Hugging Face Hub successfully at https://huggingface.co/Ghostraptor/qwen2.5-0.5b-customer-support-LoRA-dpo-merged
